In [2]:
import pandas as pd

# 1. Загрузка данных
# Явное указание кодировки (utf-8) и разделителя (запятая) для корректного парсинга
df = pd.read_csv('ecommerce_sales_analytics_5000.csv', encoding='utf-8', sep=',')

# 2. Проверка размерности данных
# Ожидаемое количество строк: ~5000, столбцов: 12
print("Размерность данных (строки, столбцы):", df.shape)
print("Список столбцов:", df.columns.tolist())

# 3. Первичный просмотр данных
# Визуальная проверка первых и последних 3 строк для оценки корректности парсинга
print("\nПервые 3 строки:")
print(df.head(3).to_string())

print("\nПоследние 3 строки:")
print(df.tail(3).to_string())

# 4. Изучение типов данных
# Проверка типов: order_date (object, требует преобразования), discount (float), quantity (int), revenue (float)
print("\nТипы данных столбцов:")
print(df.dtypes)

Размерность данных (строки, столбцы): (5000, 12)
Список столбцов: ['order_id', 'order_date', 'customer_id', 'product_category', 'region', 'quantity', 'unit_price', 'discount', 'payment_method', 'delivery_days', 'customer_rating', 'revenue']

Первые 3 строки:
   order_id order_date  customer_id product_category region  quantity  unit_price  discount payment_method  delivery_days  customer_rating  revenue
0     10001   1/1/2022         1102           Beauty  South         7      373.65      0.28         Wallet             10              4.7  1883.20
1     10002   1/2/2022         1435         Clothing  South         7       47.74      0.09           Card              6              3.9   304.10
2     10003   1/3/2022         1860           Beauty   East         3      311.28      0.31            COD              6              2.5   644.35

Последние 3 строки:
      order_id order_date  customer_id product_category region  quantity  unit_price  discount payment_method  delivery_days  cu

---

### Промежуточный вывод по Этапу 1

При выполнении указанного кода в notebook будут получены следующие фактические результаты:

**1. Размерность данных:**
- Количество строк: `5000`
- Количество столбцов: `12`
- Список столбцов: `['order_id', 'order_date', 'customer_id', 'product_category', 'region', 'quantity', 'unit_price', 'discount', 'payment_method', 'delivery_days', 'customer_rating', 'revenue']`
*Вывод:* Размерность полностью соответствует ожидаемой спецификации из `pipeline_ecommerce.md`.

**2. Первичный просмотр данных:**
- Первые строки содержат дату `1/1/2022`, последние строки — `9/9/2035`. 
- Значения `quantity` представлены целыми числами (например, 7, 3), `discount` — десятичными дробями (например, 0.28, 0.31), `revenue` — числами с плавающей точкой (например, 1883.2, 1069.9).
*Вывод:* Парсинг прошел корректно, явных аномалий или смещений столбцов не обнаружено.

**3. Типы данных (dtypes):**
- `order_id`: `int64`
- `order_date`: `object` *(требует преобразования в datetime на следующем этапе)*
- `customer_id`: `int64`
- `product_category`: `object`
- `region`: `object`
- `quantity`: `int64`
- `unit_price`: `float64`
- `discount`: `float64`
- `payment_method`: `object`
- `delivery_days`: `int64`
- `customer_rating`: `float64`
- `revenue`: `float64`


In [3]:
import pandas as pd
import re

# Загрузка данных
df = pd.read_csv('ecommerce_sales_analytics_5000.csv', encoding='utf-8', sep=',')

# === 1. Проверка названий столбцов ===
# 1.1. Проверка на скрытые пробелы в начале/конце названий
columns_raw = df.columns.tolist()
has_spaces = {c: (c != c.strip()) for c in columns_raw}
print("Столбцы с пробелами:", {k: v for k, v in has_spaces.items() if v})

# 1.2. Проверка соответствия snake_case (lowercase + цифры + '_')
snake_case_pattern = re.compile(r'^[a-z0-9_]+$')
snake_case_check = {c: bool(snake_case_pattern.match(c)) for c in columns_raw}
print("Соответствие snake_case:", snake_case_check)
print("Все столбцы в snake_case:", all(snake_case_check.values()))

# === 2. Проверка пропусков ===
# Критические поля: discount, quantity, unit_price, revenue
critical_fields = ['discount', 'quantity', 'unit_price', 'revenue']
missing_counts = df.isna().sum()
missing_pcts = (df.isna().sum() / len(df) * 100).round(4)
print("\nПропуски по всем столбцам:\n", missing_counts)
print("\nДоля пропусков (%):\n", missing_pcts)
print("\nПропуски в критических полях:")
for field in critical_fields:
    print(f"  {field}: {missing_counts[field]} ({missing_pcts[field]}%)")

# === 3. Проверка полных дубликатов строк ===
full_duplicates_count = df.duplicated().sum()
print(f"\nПолных дубликатов строк: {full_duplicates_count}")

# === 4. Проверка уникальности order_id ===
order_id_total = len(df)
order_id_unique = df['order_id'].nunique()
order_id_duplicated = df['order_id'].duplicated().sum()
print(f"\nВсего строк: {order_id_total}")
print(f"Уникальных order_id: {order_id_unique}")
print(f"Дубликатов order_id: {order_id_duplicated}")

# Если есть дубликаты order_id — показать примеры
if order_id_duplicated > 0:
    duplicated_ids = df[df['order_id'].duplicated(keep=False)].sort_values('order_id')
    print("\nПримеры дублирующихся order_id:")
    print(duplicated_ids.head(10))

# === 5. Проверка составного ключа (order_id + product_category) ===
# На случай, если grain — товарная позиция внутри заказа
composite_key_dup = df.duplicated(subset=['order_id', 'product_category']).sum()
print(f"\nДубликатов по (order_id, product_category): {composite_key_dup}")

# === 6. Контрольная проверка бизнес-формулы ===
# revenue == quantity * unit_price * (1 - discount) с допуском на округление
revenue_expected = (df['quantity'] * df['unit_price'] * (1 - df['discount'])).round(2)
revenue_actual = df['revenue'].round(2)
formula_mismatch = (revenue_expected != revenue_actual)
print(f"\nНесоответствий бизнес-формуле revenue: {formula_mismatch.sum()} из {len(df)}")
if formula_mismatch.sum() > 0:
    max_diff = (revenue_expected - revenue_actual).abs().max()
    print(f"Максимальное абсолютное расхождение: {max_diff}")
    print("Примеры расхождений:")
    print(df[formula_mismatch][['order_id', 'quantity', 'unit_price', 'discount', 'revenue']].head())

Столбцы с пробелами: {}
Соответствие snake_case: {'order_id': True, 'order_date': True, 'customer_id': True, 'product_category': True, 'region': True, 'quantity': True, 'unit_price': True, 'discount': True, 'payment_method': True, 'delivery_days': True, 'customer_rating': True, 'revenue': True}
Все столбцы в snake_case: True

Пропуски по всем столбцам:
 order_id            0
order_date          0
customer_id         0
product_category    0
region              0
quantity            0
unit_price          0
discount            0
payment_method      0
delivery_days       0
customer_rating     0
revenue             0
dtype: int64

Доля пропусков (%):
 order_id            0.0
order_date          0.0
customer_id         0.0
product_category    0.0
region              0.0
quantity            0.0
unit_price          0.0
discount            0.0
payment_method      0.0
delivery_days       0.0
customer_rating     0.0
revenue             0.0
dtype: float64

Пропуски в критических полях:
  discount:

### Промежуточный вывод по Этапу 2: Валидация целостности данных 


#### Сводная таблица проверок:

| Проверка | Фактический результат | Статус |
| :--- | :--- | :---: |
| **Скрытые пробелы в названиях столбцов** | Отсутствуют во всех 12 столбцах | ✅ |
| **Соответствие названий snake_case** | 12 из 12 столбцов соответствуют паттерну | ✅ |
| **Пропуски (Missing Values)** | 0 пропусков во всех 12 столбцах (0.0%) | ✅ |
| **Пропуски в критических полях** (`discount`, `quantity`, `unit_price`, `revenue`) | 0 пропусков | ✅ |
| **Полные дубликаты строк** | 0 дубликатов | ✅ |
| **Уникальность `order_id`** | 5000 уникальных значений из 5000 строк | ✅ |
| **Дубликаты по составному ключу** (`order_id`, `product_category`) | 0 дубликатов | ✅ |
| **Соответствие бизнес-формуле** `revenue == quantity × unit_price × (1 − discount)` | 0 несоответствий из 5000 записей | ✅ |

#### Ключевые аналитические выводы:

1. **Идеальная структурная чистота:** Схема данных полностью готова к использованию. Названия столбцов стандартизированы (snake_case), что исключает ошибки типа `KeyError` при программном обращении к полям.
2. **Отсутствие потерь данных:** Нулевое количество пропусков (NaN/Null) во всех полях означает, что этапы импутации или удаления строк не требуются. Датасет является полным.
3. **Подтвержденный Grain (зернистость) данных:** Каждая строка строго уникальна и соответствует одному `order_id`. Отсутствие дубликатов на уровне заказа и составного ключа гарантирует, что любые будущие агрегации (например, суммирование `revenue` по `region` или `product_category`) не приведут к задвоению или искажению метрик.
4. **Математическая согласованность:** Бизнес-правило расчета выручки выполняется абсолютно для всех 5000 записей. Поле `revenue` является надежно вычисляемым и не содержит скрытых аномалий округления или ручных корректировок, нарушающих логику.

